In [1]:

import os

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

for every cases after shuffling token order: true negatives that remained correctly classified (TN→TN) or became misclassified (TN→FP), and TP→FN TP→TP 
1. Use indeces of samples which extracted by notebook: "Perturbation_shuffled_tokenized_RandNeg_TN.ipynb" 
2. Find average shap value and frequencey of tokens corresponding these samples, by loading all 30 shap analysis runs results
3. Use results for statistical analysis of AT-index and shap value comarison

In [2]:
%matplotlib inline

In [2]:
path_TN_TN= "/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/TN_remain_TN_indices.csv"
TN_TN=pd.read_csv(path_TN_TN).iloc[:, 0] 
indices_TN_TN = TN_TN.to_list()
print(indices_TN_TN)

path_TN_FP= "/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/TN_turnto_FP_indices.csv"
TN_FP=pd.read_csv(path_TN_FP).iloc[:, 0] 
indices_TN_FP = TN_FP.to_list()
indices_TN_FP


[2, 4, 17, 25, 30, 32, 35, 36, 39, 44, 45, 49, 51, 52, 60, 63, 72, 90, 91, 96, 99, 101, 106, 108, 109, 117, 123, 124, 125, 128]


[0, 5, 9, 19, 24, 43, 50, 65, 68, 75, 85, 107, 113, 118]

In [ ]:
def find_tokens(indices):
    dfs = [pd.read_csv(f"//p/project1/hai_dnaori/piroozeh1/DNABERT_2/finetune/shap-values/nsample_100_aic_shuffeld/shap_values{i}.csv") for i in range(1, 30)]
    # dfs= pd.read_csv(f"/p/project1/hai_dnaori/piroozeh1/DNABERT_2/finetune/shap-values/nsample_100_aic_shuffeld/shap_values.csv")
    top_tokens_all = []
    
    cutoff=1
    for df in dfs:
            df_f = df[df["sample_index"].isin(indices)].copy()
            df_f = df_f[df_f["shap_value"].abs() < cutoff]
        
            ab = df_f[(df_f["class"] == 0) & (df_f["shap_value"] > 0)]  # add more conditions here
            
            
            top_tokens = (
                ab.groupby("token")
                .agg(
                    mean_shap=("shap_value", "mean"),
                    sum_shap=("shap_value", "sum"),
                    count=("shap_value", "count")
                    )
                .sort_values("mean_shap", ascending=False)
                .head(50)
                .reset_index())
            
            top_tokens_all.append(top_tokens)
    
    # Concatenate all top token results
    combined_top_tokens = pd.concat(top_tokens_all)
    
    # Group by token across all dataframes
    final_agg = (
        combined_top_tokens.groupby("token")
        .agg(
            freq_in_runs=("token", "count"),  # in how many runs this token was in top 30
            total_count=("count", "sum"),      # sum of counts in all runs
            avg_shap=("mean_shap", "mean")
            )
        .sort_values(["freq_in_runs", "avg_shap"], ascending=False)
                )
    
    print((final_agg.head(40)))
    
    # final_agg.to_csv("/p/project1/hai_dnaori/piroozeh1/yeast-origins/data/OriDB_random_neg/train_dev_test_oridb/DNABERT2/shuffeld_tokens_test/shap_results/TN_TN_HighestPositives_shap.csv", sep="\t", index=True)
    freq_important_tokens= final_agg.index.tolist()
    return freq_important_tokens

    
TN_TN= find_tokens(indices_TN_TN)
# print("Tokens for TN-->TN")
# print(TN_TN )
# TN_FP= find_tokens(indices_TN_FP)
# print("Tokens for TN-->FP")
# print(TN_FP )
    

         freq_in_runs  total_count  avg_shap
token                                       
CGTT               13           20  0.071716
CCTGTT             12           12  0.091655
CCTCTG             12           12  0.091548
CGG                11           55  0.065551
TTCA               11           15  0.062159
CCTT               10           15  0.086767
GAAGG              10           11  0.082725
GATAA              10           16  0.069752
CTTTCC              9            9  0.085153
TCACC               9           11  0.072679
TGATGA              9           10  0.069984
CGA                 8           13  0.087992
CCTGGTG             8            8  0.079121
CCATT               8           14  0.074684
TCTCTT              8           10  0.072729
CCTG                8           12  0.072372
CTCC                8           14  0.070638
TCACAGG             8            8  0.062237
TCCTT               7           10  0.098471
CTTTT               7            9  0.095069
CAACGG    